<a href="https://colab.research.google.com/github/uwuICQ/CV_Yandex_CourseProject/blob/main/%D0%9B%D1%83%D1%87%D1%88%D0%B0%D1%8F_%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D1%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resnet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.relu(out)
        return out

In [ ]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(5, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)

        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512, num_classes)

        self._init_weights()

    def _make_layer(self, in_channels, out_channels, blocks, stride):
        downsample = None
        if stride != 1 or in_channels != out_channels * BasicBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * BasicBlock.expansion, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels * BasicBlock.expansion)
            )
        layers = [BasicBlock(in_channels, out_channels, stride, downsample)]
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_channels * BasicBlock.expansion, out_channels))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [ ]:
# Веса
import torch
model_resnet = ResNet18(num_classes=1)
model_resnet.load_state_dict(torch.load('best_resnet.pth'))
model_resnet.eval()
state_dict = torch.load('best_resnet.pth')
for name, weight in state_dict.items():
    print(f"{name}: {weight.shape}")
    print(f"{weight}\n")

Выходные данные были обрезаны до нескольких последних строк (5000).

        [[[-0.0824, -0.1185,  0.0105],
          [ 0.0964,  0.1405, -0.0348],
          [ 0.0741,  0.0353,  0.0750]],

         [[ 0.0597,  0.1206,  0.1786],
          [-0.0005, -0.0785,  0.1068],
          [ 0.1288,  0.1014,  0.1346]],

         [[-0.0004,  0.0655, -0.0406],
          [ 0.1010, -0.0020, -0.0287],
          [-0.0613, -0.0203, -0.0921]],

         ...,

         [[ 0.0112,  0.0276,  0.0783],
          [-0.0217, -0.0084,  0.0600],
          [-0.1178, -0.1686, -0.0583]],

         [[-0.0221, -0.1126, -0.0322],
          [-0.0345, -0.0121, -0.1054],
          [-0.1623, -0.0983, -0.1361]],

         [[-0.0912,  0.0277,  0.0052],
          [-0.0712,  0.0673,  0.0827],
          [-0.0390,  0.0101,  0.1425]]],


        [[[ 0.0232, -0.1152,  0.0581],
          [-0.0429, -0.0696, -0.0205],
          [-0.0779, -0.1402, -0.0126]],

         [[ 0.0506,  0.0530, -0.0822],
          [-0.1141, -0.0074, -0.0080],
   

In [ ]:
# Инференс
model_resnet = ResNet18(num_classes=1)
model_resnet.eval()

test = torch.randn(1, 5, 256, 256)
with torch.no_grad():
    prob = torch.sigmoid(model_resnet(test))
    print(f"Предсказание: {'real' if prob > 0.5 else 'fake'}")

Предсказание: real
